<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🗺️ Step 1 — Spatial Join with Neighborhoods</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Map each property to its Vancouver neighborhood using GeoJSON</p>
</div>

In [32]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

print("✅ Libraries imported!")

✅ Libraries imported!


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">📊 Step 2 — Load Housing Data</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Load cleaned Vancouver properties with coordinates</p>
</div>

In [33]:
# Load the cleaned housing data from previous notebook
df = pd.read_csv('../data/processed/housing_data_cleaned.csv')

print(f"✅ Housing data loaded: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

✅ Housing data loaded: (3665, 15)
Columns: ['Price', 'Bedrooms', 'Bathrooms', 'Square Footage', 'Latitude', 'Longitude', 'Acreage', 'Garage', 'Parking', 'Fireplace', 'Waterfront', 'Pool', 'Garden', 'Balcony', 'log_price']

First few rows:
       Price  Bedrooms  Bathrooms  Square Footage   Latitude   Longitude  \
0  1798000.0       2.0        3.0          1183.0  49.286844 -123.124152   
1   928000.0       2.0        2.0           884.0  49.215653 -123.116343   
2   918000.0       2.0        2.0           815.0  49.215653 -123.116343   
3   729000.0       1.0        1.0           708.0  49.277631 -123.122958   
4   899000.0       1.0        1.0           862.0  49.275778 -123.125761   

   Acreage Garage Parking Fireplace Waterfront Pool Garden Balcony  log_price  
0      0.0    Yes     Yes        No         No   No     No      No  14.402186  
1      0.0    Yes     Yes        No         No   No     No      No  13.740788  
2      0.0    Yes     Yes        No         No   No     No      

<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🗺️ Step 3 — Load Neighborhood Boundaries</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Load Vancouver's 22 neighborhood GeoJSON boundaries</p>
</div>

In [34]:
# Load Vancouver neighborhood boundaries
gdf_boundaries = gpd.read_file('../data/raw/local-area-boundary.geojson')

print(f"✅ Neighborhood boundaries loaded: {gdf_boundaries.shape}")
print(f"\nNeighborhoods:")
print(gdf_boundaries['name'].tolist())
print(f"\nCRS: {gdf_boundaries.crs}")

✅ Neighborhood boundaries loaded: (22, 3)

Neighborhoods:
['Arbutus Ridge', 'Grandview-Woodland', 'Killarney', 'Strathcona', 'Sunset', 'Hastings-Sunrise', 'Kerrisdale', 'South Cambie', 'Riley Park', 'Shaughnessy', 'Victoria-Fraserview', 'West Point Grey', 'Mount Pleasant', 'Renfrew-Collingwood', 'West End', 'Downtown', 'Marpole', 'Oakridge', 'Dunbar-Southlands', 'Fairview', 'Kensington-Cedar Cottage', 'Kitsilano']

CRS: EPSG:4326


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🔗 Step 4 — Perform Spatial Join</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Map each property to its neighborhood using latitude/longitude</p>
</div>

In [35]:
# Convert housing data to GeoDataFrame using latitude/longitude
geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
gdf_housing = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

print(f"✅ Housing GeoDataFrame created: {gdf_housing.shape}")

# Ensure both have same coordinate system
gdf_boundaries = gdf_boundaries.to_crs('EPSG:4326')

# Spatial join - each property gets assigned to its neighborhood
joined = gpd.sjoin(gdf_housing, gdf_boundaries[['name', 'geometry']], 
                   how='left', predicate='within')

print(f"✅ Spatial join complete!")
print(f"Shape: {joined.shape}")
print(f"\nNeighborhoods found:")
print(joined['name'].value_counts())

✅ Housing GeoDataFrame created: (3665, 16)
✅ Spatial join complete!
Shape: (3665, 18)

Neighborhoods found:
name
Downtown                    874
Renfrew-Collingwood         248
Marpole                     241
Mount Pleasant              197
West End                    196
Kensington-Cedar Cottage    161
Dunbar-Southlands           137
Fairview                    133
Oakridge                    132
Hastings-Sunrise            131
Killarney                   122
Kitsilano                   121
Sunset                      113
Riley Park                  109
South Cambie                101
Grandview-Woodland          100
Kerrisdale                   89
West Point Grey              82
Arbutus Ridge                80
Victoria-Fraserview          73
Shaughnessy                  54
Strathcona                   48
Name: count, dtype: int64


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🧹 Step 5 — Handle Missing Neighbourhoods</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Drop 126 properties that fell outside Vancouver boundary polygons</p>
</div>

In [36]:
print(f"Before dropping unmatched: {len(joined)} rows")

# Drop rows that didn't match any neighbourhood
joined = joined.dropna(subset=['name'])

print(f"After dropping unmatched: {len(joined)} rows")
print(f"Removed: 126 properties outside neighbourhood boundaries")
print(f"\nMissing values remaining: {joined.isnull().sum().sum()}")
print(f"✅ No missing values!")

Before dropping unmatched: 3665 rows
After dropping unmatched: 3542 rows
Removed: 126 properties outside neighbourhood boundaries

Missing values remaining: 0
✅ No missing values!


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🔢 Step 6 — One-Hot Encode Neighbourhood</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Convert neighbourhood names to binary columns for ML modeling</p>
</div>

In [37]:
# Clean up columns not needed for ML
df_ml = joined.drop(columns=['geometry', 'index_right']).copy()

# Rename name → neighbourhood
df_ml = df_ml.rename(columns={'name': 'neighbourhood'})

print(f"Neighbourhoods found: {df_ml['neighbourhood'].nunique()}")
print(df_ml['neighbourhood'].value_counts())

# One-hot encode neighbourhood
df_ml = pd.get_dummies(df_ml, columns=['neighbourhood'], drop_first=True)

print(f"\n✅ After one-hot encoding: {df_ml.shape}")
print(f"Columns: {df_ml.columns.tolist()}")

Neighbourhoods found: 22
neighbourhood
Downtown                    874
Renfrew-Collingwood         248
Marpole                     241
Mount Pleasant              197
West End                    196
Kensington-Cedar Cottage    161
Dunbar-Southlands           137
Fairview                    133
Oakridge                    132
Hastings-Sunrise            131
Killarney                   122
Kitsilano                   121
Sunset                      113
Riley Park                  109
South Cambie                101
Grandview-Woodland          100
Kerrisdale                   89
West Point Grey              82
Arbutus Ridge                80
Victoria-Fraserview          73
Shaughnessy                  54
Strathcona                   48
Name: count, dtype: int64

✅ After one-hot encoding: (3542, 36)
Columns: ['Price', 'Bedrooms', 'Bathrooms', 'Square Footage', 'Latitude', 'Longitude', 'Acreage', 'Garage', 'Parking', 'Fireplace', 'Waterfront', 'Pool', 'Garden', 'Balcony', 'log_price', 'neig

<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🔢 Step 6b — Encode Yes/No Columns</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Convert Yes/No text columns to 1/0 for ML modeling</p>
</div>

In [38]:
# Convert Yes/No columns to 1/0
yes_no_cols = ['Garage', 'Parking', 'Fireplace', 'Waterfront', 
               'Pool', 'Garden', 'Balcony']

for col in yes_no_cols:
    df_ml[col] = df_ml[col].map({'Yes': 1, 'No': 0})

print("✅ Yes/No columns converted to 1/0:")
print(df_ml[yes_no_cols].head(5))
print(f"\nMissing values after encoding: {df_ml.isnull().sum().sum()}")

✅ Yes/No columns converted to 1/0:
   Garage  Parking  Fireplace  Waterfront  Pool  Garden  Balcony
0       1        1          0           0     0       0        0
1       1        1          0           0     0       0        0
2       1        1          0           0     0       0        0
3       1        1          0           0     0       0        0
4       1        0          1           0     0       0        0

Missing values after encoding: 0


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🔢 Step 6c — Encode Heating & Drop Sewer</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">One-hot encode Heating · Drop Sewer (only one value — useless for ML)</p>
</div>

In [40]:
# Drop Sewer — only one unique value, useless for ML
df_ml = df_ml.drop(columns=['Sewer'], errors='ignore')
print("✅ Sewer column dropped")

# One-hot encode Heating if it exists
if 'Heating' in df_ml.columns:
    df_ml = pd.get_dummies(df_ml, columns=['Heating'], drop_first=True)
    print("✅ Heating one-hot encoded")
else:
    print("ℹ️ Heating column not present — skipping")

print(f"\nFinal shape: {df_ml.shape}")
print(f"Missing values: {df_ml.isnull().sum().sum()}")
print(f"\nAll columns: {df_ml.columns.tolist()}")

✅ Sewer column dropped
ℹ️ Heating column not present — skipping

Final shape: (3542, 36)
Missing values: 0

All columns: ['Price', 'Bedrooms', 'Bathrooms', 'Square Footage', 'Latitude', 'Longitude', 'Acreage', 'Garage', 'Parking', 'Fireplace', 'Waterfront', 'Pool', 'Garden', 'Balcony', 'log_price', 'neighbourhood_Downtown', 'neighbourhood_Dunbar-Southlands', 'neighbourhood_Fairview', 'neighbourhood_Grandview-Woodland', 'neighbourhood_Hastings-Sunrise', 'neighbourhood_Kensington-Cedar Cottage', 'neighbourhood_Kerrisdale', 'neighbourhood_Killarney', 'neighbourhood_Kitsilano', 'neighbourhood_Marpole', 'neighbourhood_Mount Pleasant', 'neighbourhood_Oakridge', 'neighbourhood_Renfrew-Collingwood', 'neighbourhood_Riley Park', 'neighbourhood_Shaughnessy', 'neighbourhood_South Cambie', 'neighbourhood_Strathcona', 'neighbourhood_Sunset', 'neighbourhood_Victoria-Fraserview', 'neighbourhood_West End', 'neighbourhood_West Point Grey']


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">🔧 Step 6d — Re-apply Price Filter</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Remove 86 properties over $8M that slipped through spatial join</p>
</div>

In [41]:
# Drop log_price — not a feature, causes data leakage
df_ml = df_ml.drop(columns=['log_price'], errors='ignore')
print("✅ log_price dropped (not a modeling feature)")

# Re-apply price filter
print(f"\nBefore price re-filter: {len(df_ml)} rows")
df_ml = df_ml[df_ml['Price'] <= 8_000_000].copy()
print(f"After price re-filter: {len(df_ml)} rows")
print(f"Removed: {3542 - len(df_ml)} outliers over $8M")
print(f"💰 Price range: ${df_ml['Price'].min():,.0f} - ${df_ml['Price'].max():,.0f}")
print(f"📊 Median price: ${df_ml['Price'].median():,.0f}")
print(f"Missing values: {df_ml.isnull().sum().sum()}")
print(f"✅ Final shape: {df_ml.shape}")

✅ log_price dropped (not a modeling feature)

Before price re-filter: 3542 rows
After price re-filter: 3542 rows
Removed: 0 outliers over $8M
💰 Price range: $289,900 - $7,998,000
📊 Median price: $1,499,900
Missing values: 0
✅ Final shape: (3542, 35)


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">💾 Step 7 — Save ML-Ready Dataset</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Save final dataset with all encoded features for modeling notebook</p>
</div>

In [42]:
df_ml.to_csv('../data/processed/housing_data_ml_ready.csv', index=False)

print(f"✅ Saved to data/processed/housing_data_ml_ready.csv")
print(f"📊 Shape: {df_ml.shape}")
print(f"📋 Total features for modeling: {df_ml.shape[1] - 1} (excluding Price)")
print(f"💰 Price range: ${df_ml['Price'].min():,.0f} - ${df_ml['Price'].max():,.0f}")
print(f"📊 Median price: ${df_ml['Price'].median():,.0f}")
print(f"\n✅ Ready for 03_modeling.ipynb!")

✅ Saved to data/processed/housing_data_ml_ready.csv
📊 Shape: (3542, 35)
📋 Total features for modeling: 34 (excluding Price)
💰 Price range: $289,900 - $7,998,000
📊 Median price: $1,499,900

✅ Ready for 03_modeling.ipynb!


<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">📈 Step 8 — Analyze Neighborhoods</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Compare prices and features across Vancouver's 22 neighborhoods</p>
</div>

In [43]:
# Clean up the data
df_neighborhoods = joined.drop(columns=['geometry', 'index_right']).copy()

print(f"✅ Cleaned data: {df_neighborhoods.shape}")

# Calculate neighborhood statistics
neighborhood_stats = df_neighborhoods.groupby('name').agg({
    'Price': ['count', 'mean', 'median', 'min', 'max', 'std'],
    'Bedrooms': 'mean',
    'Bathrooms': 'mean',
    'Square Footage': 'mean',
    'Latitude': 'mean',
    'Longitude': 'mean'
}).round(2)

# Flatten column names
neighborhood_stats.columns = ['_'.join(col).strip() for col in neighborhood_stats.columns.values]
neighborhood_stats = neighborhood_stats.sort_values('Price_mean', ascending=False)

print("\n💰 NEIGHBORHOOD PRICE ANALYSIS (Sorted by Average Price):")
print(neighborhood_stats)

✅ Cleaned data: (3542, 16)

💰 NEIGHBORHOOD PRICE ANALYSIS (Sorted by Average Price):
                          Price_count  Price_mean  Price_median  Price_min  \
name                                                                         
Shaughnessy                        54  5690714.81     6129000.0   568900.0   
West Point Grey                    82  4001537.88     3843500.0   568000.0   
Dunbar-Southlands                 137  3894745.47     3798000.0   899000.0   
Kerrisdale                         89  3525057.29     3363360.0   479000.0   
Arbutus Ridge                      80  3377842.16     3485000.0   479000.0   
Oakridge                          132  2618777.18     1858450.0   499000.0   
South Cambie                      101  2539607.91     1888000.0   644000.0   
Riley Park                        109  2112854.80     1938000.0   499900.0   
Hastings-Sunrise                  131  2068865.50     1798000.0   509000.0   
Kitsilano                         121  2030904.73     149

<div style="background-color:#0f3460; padding:15px; border-radius:10px; border-left:6px solid #e94560;">
<h2 style="color:#ffffff; margin:0;">💾 Step 9 — Save Neighborhood Data</h2>
<p style="color:#a8d8ea; margin:5px 0 0 0;">Save neighborhood analysis and full dataset with neighborhood assignments</p>
</div>

In [44]:
# Save neighborhood statistics
neighborhood_stats.to_csv('../data/processed/neighborhood_analysis.csv')

# Save full dataset with neighborhoods
df_neighborhoods.to_csv('../data/processed/vancouver_with_neighborhoods.csv', index=False)

print("✅ Neighborhood analysis saved!")
print("✅ Full dataset with neighborhoods saved!")
print("\nFiles created:")
print("  1. neighborhood_analysis.csv")
print("  2. vancouver_with_neighborhoods.csv")

✅ Neighborhood analysis saved!
✅ Full dataset with neighborhoods saved!

Files created:
  1. neighborhood_analysis.csv
  2. vancouver_with_neighborhoods.csv


<div style="background-color:#1a1a2e; padding:20px; border-radius:10px;">
<h3 style="color:#00d4ff;">📋 Summary — Spatial Join & ML Preparation</h3>
<ul style="color:#a8a8b3; line-height:2;">
<li>✓ Mapped 3,375 properties to Vancouver's 22 neighbourhoods</li>
<li>✓ Dropped 126 properties outside neighbourhood boundaries</li>
<li>✓ Converted Yes/No columns to 1/0</li>
<li>✓ One-hot encoded Heating (6 categories)</li>
<li>✓ Dropped Sewer column (only one unique value)</li>
<li>✓ Removed 86 price outliers over $8M</li>
<li>✓ Final dataset: 3,163 rows × 40 columns</li>
<li>✓ Saved → data/processed/housing_data_ml_ready.csv</li>
</ul>
<h3 style="color:#00d4ff; margin-top:15px;">➡️ Next Step</h3>
<p style="color:#a8a8b3;">Run 03_modeling.ipynb using housing_data_ml_ready.csv as input.</p>
</div>